# HapticWay §5.6 — Custom Object Detection Model

Trains an **EfficientDet-Lite0** model (320×320, INT8) on 8 navigation-relevant classes:
`person`, `bicycle`, `bench`, `chair`, `door`, `staircase`, `pole`, `wet_floor_sign`.

**Runtime required:** GPU (T4). Runtime → Change runtime type → T4 GPU.

**Time:** ~2 hours total (dataset download ~45 min, training ~60 min on T4).

**Outputs:**
- `hapticway_custom.tflite` — INT8 quantized EfficientDet-Lite0
- `hapticway_labels.txt` — 9-line label file (??? + 8 classes)

Place both files in `assets/models/` and `assets/labels/` respectively,
then update `camera_isolate.dart` as noted in the §5.6 comment.

## 1 · Install dependencies

In [ ]:
# Pin versions for reproducibility. TFLite Model Maker 0.4.3 is the last
# stable release that targets TF 2.13 and produces TFLite_Detection_PostProcess
# compatible output tensors (same format as the stock SSD MobileNet V1 model).
!pip install -q tflite-model-maker==0.4.3
!pip install -q fiftyone==0.23.8
!pip install -q pycocotools

In [ ]:
import os, json, shutil
from pathlib import Path
import tensorflow as tf

print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2 · Download training data from Open Images v7

Uses Google's Open Images v7 dataset — no account needed.
Downloads bounding-box annotations for 7 of the 8 target classes.
`wet_floor_sign` is handled separately in §3.

In [ ]:
import fiftyone as fo
import fiftyone.zoo as foz

# Open Images class names → HapticWay app names
# The app's postprocess.dart checks against these exact strings.
CLASS_MAP = {
    'Person':   'person',
    'Bicycle':  'bicycle',
    'Bench':    'bench',
    'Chair':    'chair',
    'Door':     'door',
    'Stairs':   'staircase',
    'Pole':     'pole',
}
OI_CLASSES = list(CLASS_MAP.keys())

N_TRAIN = 300   # images per class for training
N_VAL   = 60    # images per class for validation

print('Downloading training split...')
oi_train = foz.load_zoo_dataset(
    'open-images-v7',
    split='train',
    label_types=['detections'],
    classes=OI_CLASSES,
    max_samples=N_TRAIN * len(OI_CLASSES),
    dataset_name='hapticway_oi_train',
)
print(f'Train samples: {len(oi_train)}')

print('Downloading validation split...')
oi_val = foz.load_zoo_dataset(
    'open-images-v7',
    split='validation',
    label_types=['detections'],
    classes=OI_CLASSES,
    max_samples=N_VAL * len(OI_CLASSES),
    dataset_name='hapticway_oi_val',
)
print(f'Val samples: {len(oi_val)}')

## 3 · Download wet_floor_sign data from Roboflow (optional)

Sign up at https://roboflow.com (free). Go to
https://universe.roboflow.com and search **"wet floor sign detection"**.
Pick a dataset with ≥200 images, click **Download → COCO format**,
copy the download snippet and paste it below.

If you skip this cell, the model will not detect `wet_floor_sign`.
The app's `_targetClasses` filter will simply never match that label.

In [ ]:
# OPTIONAL — uncomment and fill in your Roboflow snippet:
#
# !pip install -q roboflow
# from roboflow import Roboflow
# rf = Roboflow(api_key='YOUR_API_KEY')
# project = rf.workspace('YOUR_WORKSPACE').project('wet-floor-sign-detection')
# version = project.version(1)
# rf_dataset = version.download('coco', location='/content/rf_wfs')
#
# After downloading, set:
# WFS_TRAIN_DIR  = '/content/rf_wfs/train'
# WFS_TRAIN_JSON = '/content/rf_wfs/train/_annotations.coco.json'
# WFS_VAL_DIR    = '/content/rf_wfs/valid'
# WFS_VAL_JSON   = '/content/rf_wfs/valid/_annotations.coco.json'
# HAVE_WFS = True

HAVE_WFS = False  # set True after uncommenting above

## 4 · Export to COCO JSON and remap class names

In [ ]:
TRAIN_DIR = Path('/content/hapticway_data/train')
VAL_DIR   = Path('/content/hapticway_data/val')

def export_and_remap(fo_dataset, export_dir: Path, class_map: dict):
    """Export fiftyone dataset to COCO JSON, renaming classes via class_map."""
    export_dir.mkdir(parents=True, exist_ok=True)
    images_dir = export_dir / 'images'
    images_dir.mkdir(exist_ok=True)
    anno_path = export_dir / 'labels.json'

    fo_dataset.export(
        export_dir=str(export_dir),
        dataset_type=fo.types.COCODetectionDataset,
        label_field='ground_truth',
        classes=list(class_map.keys()),
    )

    # Remap category names to HapticWay app names
    with open(anno_path) as f:
        coco = json.load(f)
    for cat in coco['categories']:
        cat['name'] = class_map.get(cat['name'], cat['name'])
    with open(anno_path, 'w') as f:
        json.dump(coco, f)

    return str(images_dir), str(anno_path)


print('Exporting train set...')
train_img_dir, train_anno = export_and_remap(oi_train, TRAIN_DIR, CLASS_MAP)

print('Exporting val set...')
val_img_dir, val_anno = export_and_remap(oi_val, VAL_DIR, CLASS_MAP)

print('Done.')

In [ ]:
# Merge wet_floor_sign annotations into the COCO JSONs (if downloaded)
if HAVE_WFS:
    def merge_coco(base_json_path, extra_json_path, extra_images_dir, base_images_dir):
        with open(base_json_path) as f:
            base = json.load(f)
        with open(extra_json_path) as f:
            extra = json.load(f)

        # Remap IDs to avoid collisions
        max_img_id  = max(i['id'] for i in base['images']) if base['images'] else 0
        max_ann_id  = max(a['id'] for a in base['annotations']) if base['annotations'] else 0
        max_cat_id  = max(c['id'] for c in base['categories'])

        # Add new category
        wfs_cat_id = max_cat_id + 1
        base['categories'].append({'id': wfs_cat_id, 'name': 'wet_floor_sign'})

        # Copy images
        img_id_map = {}
        for img in extra['images']:
            old_id = img['id']
            new_id = max_img_id + old_id
            img_id_map[old_id] = new_id
            img['id'] = new_id
            src = Path(extra_images_dir) / img['file_name']
            dst = Path(base_images_dir) / img['file_name']
            if src.exists():
                shutil.copy(str(src), str(dst))
            base['images'].append(img)

        # Copy annotations, remap to new category/image IDs
        for ann in extra['annotations']:
            ann['id'] += max_ann_id
            ann['image_id'] = img_id_map.get(ann['image_id'], ann['image_id'])
            ann['category_id'] = wfs_cat_id
            base['annotations'].append(ann)

        with open(base_json_path, 'w') as f:
            json.dump(base, f)
        print(f'Merged {len(extra["images"])} wet_floor_sign images')

    merge_coco(train_anno, WFS_TRAIN_JSON, WFS_TRAIN_DIR + '/images', train_img_dir)
    merge_coco(val_anno,   WFS_VAL_JSON,   WFS_VAL_DIR   + '/images', val_img_dir)
else:
    print('Skipping wet_floor_sign merge (HAVE_WFS=False)')

## 5 · Train EfficientDet-Lite0 via TFLite Model Maker

In [ ]:
from tflite_model_maker import object_detector
from tflite_model_maker.config import ExportFormat, QuantizationConfig

train_data = object_detector.DataLoader.from_coco(
    images_dir=train_img_dir,
    annotation_path=train_anno,
)
val_data = object_detector.DataLoader.from_coco(
    images_dir=val_img_dir,
    annotation_path=val_anno,
)

print('Training classes:', train_data.label_map)
print('Train images:', len(train_data))
print('Val images:  ', len(val_data))

In [ ]:
# EfficientDet-Lite0: 320×320 input, ~4M parameters, ~1.9ms on Pixel 4.
# do_fine_tuning=True retrains all layers (not just the head).
# Increase epochs to 100 if mAP is below 0.40 after first run.
spec = object_detector.EfficientDetSpec(
    model_name='efficientdet-lite0',
    hparams={
        'max_instances_per_image': 100,
        'max_detections': 10,
    },
    model_dir='/content/model_checkpoint',
    epochs=60,
    batch_size=8,
    do_fine_tuning=True,
)

model = object_detector.create(
    train_data,
    model_spec=spec,
    val_data=val_data,
    epochs=60,
    batch_size=8,
    train_whole_model=True,
)
print('Training complete.')

## 6 · Evaluate on validation set

In [ ]:
# Reports COCO-style mAP. Target: mAP@[0.50:0.95] ≥ 0.35 for a usable model.
# If below 0.25, increase epochs (cell above) and retrain.
metrics = model.evaluate(val_data)
print('Validation mAP:', metrics)

## 7 · Export INT8 quantized TFLite model

In [ ]:
OUTPUT_DIR = Path('/content/hapticway_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# INT8 quantization using training data as calibration set.
# This matches the current stock model's quantisation level.
model.export(
    export_dir=str(OUTPUT_DIR),
    tflite_filename='hapticway_custom.tflite',
    quantization_config=QuantizationConfig.for_int8(train_data),
)

print('Model size:', (OUTPUT_DIR / 'hapticway_custom.tflite').stat().st_size // 1024, 'KB')

In [ ]:
# Write the label file in HapticWay format:
# Line 0 = ??? (background placeholder, keeps postprocess.dart's classIdx+1 offset correct)
# Lines 1-N = class names in the order the model uses them

# Retrieve class order from the trained model's label map
label_map = train_data.label_map  # dict: int_id → class_name
# label_map is 1-indexed (1=first class). Sort by key.
ordered_labels = [label_map[k] for k in sorted(label_map.keys())]

labels_out = OUTPUT_DIR / 'hapticway_labels.txt'
with open(labels_out, 'w') as f:
    f.write('???\n')  # index 0 — background placeholder
    for label in ordered_labels:
        f.write(label + '\n')

print('Label file written:')
print(labels_out.read_text())

## 8 · Download outputs to your PC

Run this cell to download both files. Then:
1. Copy `hapticway_custom.tflite` → `assets/models/hapticway_custom.tflite`
2. Copy `hapticway_labels.txt` → `assets/labels/hapticway_labels.txt` (overwrite the placeholder)
3. In `lib/inference/camera_isolate.dart`, replace the two asset paths as noted in the §5.6 comment
4. Run `flutter run` to test on device

In [ ]:
from google.colab import files

files.download(str(OUTPUT_DIR / 'hapticway_custom.tflite'))
files.download(str(OUTPUT_DIR / 'hapticway_labels.txt'))

## 9 · Verify output tensor format (optional sanity check)

Confirms the exported model has the same 4-tensor output format as the stock SSD MobileNet V1,
so `tflite_runner.dart` and `postprocess.dart` need no changes.

In [ ]:
import numpy as np

interpreter = tf.lite.Interpreter(model_path=str(OUTPUT_DIR / 'hapticway_custom.tflite'))
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('=== INPUT ===')
for d in input_details:
    print(f"  [{d['index']}] {d['name']}  shape={d['shape']}  dtype={d['dtype']}")

print('=== OUTPUTS ===')
for d in output_details:
    print(f"  [{d['index']}] {d['name']}  shape={d['shape']}  dtype={d['dtype']}")

# Expected:
#   Input  [0]  shape=[1, 320, 320, 3]  dtype=uint8
#   Output [0]  boxes   shape=[1, 10, 4]
#   Output [1]  classes shape=[1, 10]
#   Output [2]  scores  shape=[1, 10]
#   Output [3]  count   shape=[1]

# Run a dummy inference to confirm no errors
dummy = np.zeros((1, 320, 320, 3), dtype=np.uint8)
interpreter.set_tensor(input_details[0]['index'], dummy)
interpreter.invoke()
print('\nDummy inference OK.')